In [1]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
import bs4 



USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")


In [3]:
loader = WebBaseLoader(
    web_path=("https://www.geeksforgeeks.org/cpp/c-plus-plus/"),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("MainArticleContent_articleMainContentCss__b_1_R article--viewer_content")
        )
    ),

)

docs =loader.load()

In [4]:
## Converting document to chunks

from langchain_text_splitters import  RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)


In [5]:
chunks

[Document(metadata={'source': 'https://www.geeksforgeeks.org/cpp/c-plus-plus/'}, page_content='C++ is a programming language known for its fast speed, low level memory management and is often taught as first programming language.Used for making operating systems, embedded systems, graphical user interfaces and nowadays in High Frequency Trading (HFT) systems.Supports both low-level and high-level features\xa0such as manual memory management and Object Oriented Programming respectively.Syntax similarity\xa0with C, Java, and C# makes it easier to switch languages.Provides one of the\xa0fastest execution speeds\xa0among high level languages, which can be a deciding factor in Competitive Programming or high-performance applications.BasicsThis section guides you through the basic concepts of C++ programming. It covers topics that teaches you to write your first program, manage data, perform different operations and control the flow of the program.IntroductionInstalling C++ CompilerSetting u

In [6]:
embeddings = HuggingFaceEmbeddings(model='BAAI/bge-base-en-v1.5')


c:\Users\yash\Desktop\genai\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6357.23it/s]


In [7]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=chunks , embedding=embeddings)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)


In [8]:
retriever.invoke('stl')

[Document(id='ba06d161-335f-4e52-b086-21225775182b', metadata={'source': 'https://www.geeksforgeeks.org/cpp/c-plus-plus/'}, page_content='of dynamic memory management.new and delete Memory LeakQuiz: Dynamic Memory ManagementObject Oriented Programming (OOP) This section covers key concepts of Object-Oriented Programming (OOP) in C++ such as classes, objects, encapsulation, inheritance, polymorphism, and abstraction.Object Oriented Programming (OOP)Classes and ObjectsConstructorsEncapsulationPolymorphismInheritanceAbstractionQuiz: OOP QuizStandard Template Library (STL)This section covers Standard Template Library (STL) which is an in-built library that provides a set of commonly used data structures such as vectors, lists, stacks, queues, maps, etc. and algorithms that enhance productivity and performance.TemplatesStandard Template Library (STL)AlgorithmsContainersIteratorsVectorStackQueueMapSetQuiz: STL QuizException Handling Exception handling are the techniques to handle runtime err

In [9]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , system_prompt),
        ("human" ,'{input}')
    ]
)

In [10]:
llm = ChatGroq(model  = 'groq/compound')

In [11]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain


question_answer_chain = create_stuff_documents_chain(llm , prompt)
rag_chain = create_retrieval_chain(retriever , question_answer_chain)

In [12]:
response = rag_chain.invoke({"input" :  "What is STL ? "})

In [13]:
response['answer']

'The Standard Template Library (STL) is a C++ library of generic, template‑based components that provide ready‑made data structures (containers such as\u202fvector, list, map, stack, queue) and algorithms (sorting, searching, etc.) together with iterators to traverse them. These components are highly reusable, type‑safe, and efficiently implemented, allowing developers to write concise, high‑performance code without reinventing common data‑handling tools. The STL forms a core part of the C++ Standard Library.'

In [16]:
response = rag_chain.invoke({"input" : "Tell me more about it"})

In [18]:
## so here we dont have the context of the previous chat , model dont know what 'it' refers to
response

{'input': 'Tell me more about it',
 'context': [Document(id='e069142b-c2ad-4837-8cc6-64e2ea01e4bc', metadata={'source': 'https://www.geeksforgeeks.org/cpp/c-plus-plus/'}, page_content='SynchronizationAdvanced ConceptsThis section covers advanced concepts in C++ such as move preprocessor, etc. Mastering these topics allows developers to write efficient, high-performance C++ applications.PreprocessorNamespacesSmart PointersCallbacksSignal HandlingSkill AssessmentsTest what you have learnt through this C++ using a series of our Skill Assessment Test.Beginner Skill Assessment Practice TestIntermediate Skill Assessment Practice TestAdvanced Skill Assessment Practice TestInterview QuestionsQuickly prepare yourself for C++ interviews with the help of our carefully curated list of commonly asked interview questions.C++ Interview Questions and Answers Important LinksComparison of various programming languagesCareers and jobs in C++Companies that use C++'),
  Document(id='ee00a5f0-f483-4de7-adfd

## Adding Chat History

In [19]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [("system" , contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human" , '{input}')]
)



In [20]:
history_aware_retriever = create_history_aware_retriever(llm , retriever , contextualize_q_prompt)
## This will help to make the Standalone Question

In [21]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm , qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever , question_answer_chain)

In [22]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id : str ) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [23]:
conversational_rag_chain = RunnableWithMessageHistory(  ## to fetch the chat history
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [25]:
conversational_rag_chain.invoke(
    {"input": "What is STL?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)

{'input': 'What is STL?',
 'chat_history': [],
 'context': [Document(id='ba06d161-335f-4e52-b086-21225775182b', metadata={'source': 'https://www.geeksforgeeks.org/cpp/c-plus-plus/'}, page_content='of dynamic memory management.new and delete Memory LeakQuiz: Dynamic Memory ManagementObject Oriented Programming (OOP) This section covers key concepts of Object-Oriented Programming (OOP) in C++ such as classes, objects, encapsulation, inheritance, polymorphism, and abstraction.Object Oriented Programming (OOP)Classes and ObjectsConstructorsEncapsulationPolymorphismInheritanceAbstractionQuiz: OOP QuizStandard Template Library (STL)This section covers Standard Template Library (STL) which is an in-built library that provides a set of commonly used data structures such as vectors, lists, stacks, queues, maps, etc. and algorithms that enhance productivity and performance.TemplatesStandard Template Library (STL)AlgorithmsContainersIteratorsVectorStackQueueMapSetQuiz: STL QuizException Handling 

In [31]:
get_session_history('abc123')

InMemoryChatMessageHistory(messages=[HumanMessage(content='What is STL?', additional_kwargs={}, response_metadata={}), AIMessage(content='The STL (Standard Template Library) is a built‑in C++ library that supplies generic containers—such as vectors, lists, stacks, queues, maps, and sets—and a rich set of algorithms to operate on them. These components are templated, allowing them to work with any data type while providing high performance and code reuse. By using the STL, developers can write more productive and efficient programs without implementing common data structures and algorithms from scratch.', additional_kwargs={}, response_metadata={})])

In [29]:
conversational_rag_chain.invoke(
    {"input": "Tell me more about it?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)

APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}

In [ ]:
conversational_rag_chain.invoke(
    {"input": "What Question did i asked ?"},
    config={
        "configurable": {"session_id": "abc12"}
    },  # constructs a key "abc123" in `store`.
)['answer']

'Your original question was: **“What Question did I ask?”**'

In [ ]:
conversational_rag_chain.invoke(
    {"input": "What Question did i asked ?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)['answer']

'Your original question was:\n\n**“What is STL?”** \n\nYou later asked for more detail (“Tell me more about it?”), and I responded with an overview of the Standard Template Library—its containers, algorithms, iterators, why it’s useful, and a short code example.'

In [ ]:
from langchain.agents import crea